This notebook is set up to postprocess the raw data and devise statistics for given polygons. 

In [ ]:
import os
import requests
import py7zr
import geopandas as gpd
import pandas as pd
import rasterio
import xarray as xr
import rioxarray as rxr
from rioxarray.merge import merge_arrays
import numpy as np
from scipy import ndimage
from exactextract import exact_extract

In [ ]:
# set variables
data_dir = "../data"

raster_dir = data_dir + "/grasslvnd/grassland_age"
gdb_path = data_dir + "/LAD_2005_izejas.gdb"
lauki_path = data_dir + "/combined_lauki.gpkg"

In [ ]:
# url for the detailed administrative boundaries dataset
admin_url_detailed = "https://data.gov.lv/dati/dataset/0b5279c2-acac-438f-9dd9-25d82e6834d8/resource/85aab9e3-a65a-42c0-ab8f-be249f96c58d/download/admterapdzviet.7z"

In [ ]:
# download and extract the detailed administrative boundaries dataset
adm_ter = admin_url_detailed.split("/")[-1].split(".")[0]  
archive_path = os.path.join(data_dir, f"data_{adm_ter}.7z")
extract_path = os.path.join(data_dir, f"data_{adm_ter}")
os.makedirs(extract_path, exist_ok=True)

response = requests.get(admin_url_detailed)
response.raise_for_status()

with open(archive_path, 'wb') as f:
    f.write(response.content)

with py7zr.SevenZipFile(archive_path, mode='r') as archive:
    archive.extractall(path=extract_path)

In [ ]:
# find all shapefiles in the extracted data directory
shapefiles = []
for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.endswith(".shp"):
            shapefiles.append(os.path.join(root, file))
# select only those shapefiles that contain "pagasti" or "pilsetas" in their name
use_files = [f for f in shapefiles if "pagasti" in f.lower() or "pilsetas" in f.lower()]
print(f"{len(use_files)} shapefiles.")

In [ ]:
# merge the selected shapefiles into a single GeoDataFrame
gdfs = []
for shp in use_files:
    gdf = gpd.read_file(shp)
    gdfs.append(gdf)

merged_gdf = pd.concat(gdfs, ignore_index=True)

In [ ]:
# read the raster files and merge them into a single xarray DataArray
raster_files = [os.path.join(raster_dir, f) for f in os.listdir(raster_dir) if f.endswith(".tif")]
rasters = [rxr.open_rasterio(fp, chunks={"x": 1024, "y": 1024}) for fp in raster_files]
merged = merge_arrays(rasters)
merged = merged.squeeze()

In [ ]:
# write raster to disk
raw_raster = os.path.join(data_dir, "raw_raster.tif")
merged.rio.to_raster(raw_raster)

In [ ]:
# function to perform sieve operation on a raster block, set to remove connected components smaller than 10 pixels
def sieve_block(block, min_size=10, connectivity=1):
    data = np.array(block)

    if data.ndim == 3:
        data = data[0, :, :]
    unique_vals = np.unique(data)
    for val in unique_vals:
        if val == 0:
            continue
        mask = data == val
        labeled, _ = ndimage.label(
            mask,
            structure=ndimage.generate_binary_structure(2, connectivity)
        )
        sizes = np.bincount(labeled.ravel())
        remove = sizes < min_size
        data[remove[labeled]] = 0
    return data

In [ ]:
# create a binary mask of the merged raster where non-zero values are set to 1
binary = (merged > 0).astype("uint8")
# apply the sieve operation to the binary mask of the merged raster using xarray's apply_ufunc
sieved_io = xr.apply_ufunc(
    sieve_block,
    binary,
    kwargs={"min_size": 10, "connectivity": 1},
    input_core_dims=[["y", "x"]],
    output_core_dims=[["y", "x"]],
    vectorize=False,
    dask="parallelized",
    output_dtypes="uint8",
)

# get the original values from the merged raster using the sieved binary mask
sieved = sieved_io * merged

# write the sieved raster to disk
sieved_raster = os.path.join(data_dir, "sieved_raster.tif")
sieved.rio.to_raster(sieved_raster)

In [ ]:
# read the 2005 administrative boundaries from the geodatabase, explode multipart geometries, remove empty and duplicate geometries
lad_2005 = gpd.read_file(gdb_path)
lad_2005 = lad_2005.explode(index_parts=False)
lad_2005 = lad_2005[~lad_2005.geometry.is_empty]
lad_2005 = lad_2005.drop_duplicates(subset='geometry')

# clip the sieved raster to the 2005 administrative boundaries, keeping all pixels that touch the boundaries
sieved_2005 = sieved.rio.clip(lad_2005.geometry, lad_2005.crs, drop=False, invert=False, all_touched=True)

# write the sieved and masked raster to disk
sieved_masked_raster = os.path.join(data_dir, "sieved_masked_raster.tif")
sieved_2005.rio.to_raster(sieved_masked_raster)

In [ ]:
# read the field data from the GeoPackage, explode multipart geometries, remove empty and duplicate geometries
field_data = gpd.read_file(lauki_path)
field_data = field_data.explode(index_parts=False)
field_data = field_data[~field_data.geometry.is_empty]
field_data = field_data.drop_duplicates(subset='geometry')

# mask out the field data
sieved_2005_nodecl = sieved_2005.rio.clip(field_data.geometry, field_data.crs, drop=False, invert=True, all_touched=True)

# write the sieved and masked raster to disk
sieved_masked_nodecl_raster = os.path.join(data_dir, "undeclared_raster.tif")
sieved_2005_nodecl.rio.to_raster(sieved_masked_nodecl_raster)

In [ ]:
# create a binary mask of the sieved and masked raster where non-zero values are set to 1
binary_nodecl = (sieved_2005_nodecl > 0).astype("uint8")
# sieve the raster again to remove small components that may have been created by the masking operation
sieved_nodecl_io = xr.apply_ufunc(
    sieve_block,
    binary_nodecl,
    kwargs={"min_size": 10, "connectivity": 1},
    input_core_dims=[["y", "x"]],
    output_core_dims=[["y", "x"]],
    vectorize=False,
    dask="parallelized",
    output_dtypes="uint8",
)

# get the original values from the sieved and masked raster using the sieved binary mask
sieved_nodecl = sieved_nodecl_io * sieved_2005_nodecl

# write the final sieved and masked raster to disk
final_raster = os.path.join(data_dir, "undeclared_raster_sieved.tif")
sieved_nodecl.rio.to_raster(final_raster)

In [ ]:
# create a results GeoPackage to store the results
results_gpkg = os.path.join(data_dir, "results_grassland_analysis.gpkg")

In [ ]:
# perform zonal statistics on the raw raster
out_path = os.path.join(data_dir, "pilsetas_pagasti_grass_raw.parquet")
df = exact_extract(raw_raster, merged_gdf, ["unique", "frac"], progress=True, output='pandas')
df_exploded = df.apply(pd.Series.explode)
pivoted_df = df_exploded.pivot(columns='unique', values='frac')
pivoted_df = pivoted_df.reset_index(drop=True)
pivoted_df.columns = [f"age_{col}" for col in pivoted_df.columns]

adm_gdf = merged_gdf.join(pivoted_df)
adm_gdf["area_pag"] = round(adm_gdf.geometry.area / 10000, 2)
for col in pivoted_df.columns:
    adm_gdf[col] = (adm_gdf[col] * adm_gdf["area_pag"])
    adm_gdf[col] = pd.to_numeric(adm_gdf[col], errors="coerce").round(2).fillna(0)

grass_cols = [c for c in pivoted_df.columns if c != "age_0"] # exclude age_0 (non-grass) from grass_cols
adm_gdf["area_grass"] = adm_gdf[grass_cols].sum(axis=1).round(2)
adm_gdf.to_parquet(out_path)
adm_gdf.to_file(results_gpkg, layer="pilsetas_pagasti_grass_raw", driver="GPKG")
adm_gdf.to_csv(os.path.join(data_dir, "pilsetas_pagasti_grass_raw.csv"), 
               columns=["CODE", "LABEL", "age_1", "age_2", "age_3", "age_4", "age_5", "age_6", "age_7", "age_8", "area_grass", "area_pag"], 
               index=False, 
               encoding="utf-8-sig")

In [ ]:
# perform zonal statistics on the sieved raster
out_path = os.path.join(data_dir, "pilsetas_pagasti_grass_sieved.parquet")
df = exact_extract(sieved_raster, merged_gdf, ["unique", "frac"], progress=True, output='pandas')
df_exploded = df.apply(pd.Series.explode)
pivoted_df = df_exploded.pivot(columns='unique', values='frac')
pivoted_df = pivoted_df.reset_index(drop=True)
pivoted_df.columns = [f"age_{col}" for col in pivoted_df.columns]

adm_gdf = merged_gdf.join(pivoted_df)
adm_gdf["area_pag"] = round(adm_gdf.geometry.area / 10000, 2)
for col in pivoted_df.columns:
    adm_gdf[col] = (adm_gdf[col] * adm_gdf["area_pag"])
    adm_gdf[col] = pd.to_numeric(adm_gdf[col], errors="coerce").round(2).fillna(0)

grass_cols = [c for c in pivoted_df.columns if c != "age_0"] # exclude age_0 (non-grass) from grass_cols
adm_gdf["area_grass"] = adm_gdf[grass_cols].sum(axis=1).round(2)
adm_gdf.to_parquet(out_path)
adm_gdf.to_file(results_gpkg, layer="pilsetas_pagasti_grass_sieved", driver="GPKG")
adm_gdf.to_csv(os.path.join(data_dir, "pilsetas_pagasti_grass_sieved.csv"), 
               columns=["CODE", "LABEL", "age_1", "age_2", "age_3", "age_4", "age_5", "age_6", "age_7", "age_8", "area_grass", "area_pag"], 
               index=False, 
               encoding="utf-8-sig")

In [ ]:
# perform zonal statistics on the sieved and masked raster
out_path = os.path.join(data_dir, "pilsetas_pagasti_grass_sieved_masked.parquet")
df = exact_extract(sieved_masked_raster, merged_gdf, ["unique", "frac"], progress=True, output='pandas')
df_exploded = df.apply(pd.Series.explode)
pivoted_df = df_exploded.pivot(columns='unique', values='frac')
pivoted_df = pivoted_df.reset_index(drop=True)
pivoted_df.columns = [f"age_{col}" for col in pivoted_df.columns]

adm_gdf = merged_gdf.join(pivoted_df)
adm_gdf["area_pag"] = round(adm_gdf.geometry.area / 10000, 2)
for col in pivoted_df.columns:
    adm_gdf[col] = (adm_gdf[col] * adm_gdf["area_pag"])
    adm_gdf[col] = pd.to_numeric(adm_gdf[col], errors="coerce").round(2).fillna(0)

grass_cols = [c for c in pivoted_df.columns if c != "age_0"] # exclude age_0 (non-grass) from grass_cols
adm_gdf["area_grass"] = adm_gdf[grass_cols].sum(axis=1).round(2)
adm_gdf.to_parquet(out_path)
adm_gdf.to_file(results_gpkg, layer="pilsetas_pagasti_grass_sieved_masked", driver="GPKG")
adm_gdf.to_csv(os.path.join(data_dir, "pilsetas_pagasti_grass_sieved_masked.csv"), 
               columns=["CODE", "LABEL", "age_1", "age_2", "age_3", "age_4", "age_5", "age_6", "age_7", "age_8", "area_grass", "area_pag"], 
               index=False, 
               encoding="utf-8-sig")

In [ ]:
# perform zonal statistics on the sieved and masked raster with declared fields removed
out_path = os.path.join(data_dir, "pilsetas_pagasti_grass_undeclared.parquet")
df = exact_extract(sieved_masked_nodecl_raster, merged_gdf, ["unique", "frac"], progress=True, output='pandas')
df_exploded = df.apply(pd.Series.explode)
pivoted_df = df_exploded.pivot(columns='unique', values='frac')
pivoted_df = pivoted_df.reset_index(drop=True)
pivoted_df.columns = [f"age_{col}" for col in pivoted_df.columns]

adm_gdf = merged_gdf.join(pivoted_df)
adm_gdf["area_pag"] = round(adm_gdf.geometry.area / 10000, 2)
for col in pivoted_df.columns:
    adm_gdf[col] = (adm_gdf[col] * adm_gdf["area_pag"])
    adm_gdf[col] = pd.to_numeric(adm_gdf[col], errors="coerce").round(2).fillna(0)

grass_cols = [c for c in pivoted_df.columns if c != "age_0"] # exclude age_0 (non-grass) from grass_cols
adm_gdf["area_grass"] = adm_gdf[grass_cols].sum(axis=1).round(2)
adm_gdf.to_parquet(out_path)
adm_gdf.to_file(results_gpkg, layer="pilsetas_pagasti_grass_undeclared", driver="GPKG")
adm_gdf.to_csv(os.path.join(data_dir, "pilsetas_pagasti_grass_undeclared.csv"), 
               columns=["CODE", "LABEL", "age_1", "age_2", "age_3", "age_4", "age_5", "age_6", "age_7", "age_8", "area_grass", "area_pag"], 
               index=False, 
               encoding="utf-8-sig")

In [ ]:
# perform zonal statistics on the sieved and masked raster with declared fields removed
out_path = os.path.join(data_dir, "pilsetas_pagasti_grass_undeclared_sieved.parquet")
df = exact_extract(final_raster, merged_gdf, ["unique", "frac"], progress=True, output='pandas')
df_exploded = df.apply(pd.Series.explode)
pivoted_df = df_exploded.pivot(columns='unique', values='frac')
pivoted_df = pivoted_df.reset_index(drop=True)
pivoted_df.columns = [f"age_{col}" for col in pivoted_df.columns]

adm_gdf = merged_gdf.join(pivoted_df)
adm_gdf["area_pag"] = round(adm_gdf.geometry.area / 10000, 2)
for col in pivoted_df.columns:
    adm_gdf[col] = (adm_gdf[col] * adm_gdf["area_pag"])
    adm_gdf[col] = pd.to_numeric(adm_gdf[col], errors="coerce").round(2).fillna(0)

grass_cols = [c for c in pivoted_df.columns if c != "age_0"] # exclude age_0 (non-grass) from grass_cols
adm_gdf["area_grass"] = adm_gdf[grass_cols].sum(axis=1).round(2)
adm_gdf.to_parquet(out_path)
adm_gdf.to_file(results_gpkg, layer="pilsetas_pagasti_grass_undeclared_sieved", driver="GPKG")
adm_gdf.to_csv(os.path.join(data_dir, "pilsetas_pagasti_grass_undeclared_sieved.csv"), 
               columns=["CODE", "LABEL", "age_1", "age_2", "age_3", "age_4", "age_5", "age_6", "age_7", "age_8", "area_grass", "area_pag"], 
               index=False, 
               encoding="utf-8-sig")